In [ ]:
from arraylake import Client
import xarray as xr
client = Client()

era5_repo = client.get_repo("earthmover-public/era5-private")
era5_session = era5_repo.readonly_session("main")
era5 = xr.open_zarr(era5_session.store, group="single/temporal", chunks={}).sel(valid_time=slice("2015","2026"))[["t2m","d2m","tp"]]

In [ ]:
match_location = "Boston"
match_lon = -71.06
match_lat = 42.36

In [ ]:
%%time
e = era5.t2m.sel(longitude=match_lon, latitude=match_lat, method="nearest").load()
e

In [ ]:
#forecast_repo = client.get_repo("spring-data/ecmwf-ifs-15-day-forecast-low-latency-subscription")
forecast_repo = client.get_repo("spring-data/ecwmf-ifs-15-days-forecast-open")
forecast_session = forecast_repo.readonly_session("main")
forecast = xr.open_zarr(forecast_session.store, chunks={})

In [ ]:
%%time
f = forecast["2t"].sel(longitude=match_lon, latitude=match_lat, method="nearest", step=["0 days", "1 days", "2 days"]).isel(time=slice(-7*4,None)).load()
f

In [ ]:
f["valid_time"] = f.step + f.time

In [ ]:
for s in f.step:
    f.sel(step=s).plot(x="valid_time")#label=f"step {int(s)}")
e.sel(valid_time="2026-06").plot(c='k',lw=2)
#plt.label()

In [ ]:
%time reanalysis = forecast["2t"].sel(longitude=match_lon, latitude=match_lat, method="nearest", lead="0 days").load()